In [1]:
import numpy as np
import pandas as pd
import re
import string
import pickle



In [2]:
text = "great i love product"

In [3]:
def remove_punctuations(text):
    for punctuations in string.punctuation:
        text = text.replace(punctuations,'')
    return text



In [4]:
with open('../static/model/corpora/stopwords/english','r') as file:
    sw = file.read().splitlines()



In [5]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()


In [6]:
def preprocessing(text):
    data=pd.DataFrame([text],columns=['tweet'])

    data["tweet"]=data["tweet"].apply(lambda x:" ".join(x.lower() for x in x.split() ))
    data["tweet"]=data["tweet"].apply(lambda x:" ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x, flags=re.MULTILINE) for x in x.split()))
    data["tweet"]=data["tweet"].apply(remove_punctuations)
    data["tweet"]=data["tweet"].str.replace('\d','',regex=True)
    data["tweet"]=data["tweet"].apply(lambda x:" ".join(x for x in x.split() if x not in sw))
    data["tweet"]=data["tweet"].apply(lambda x : " ".join(ps.stem(x) for x in x.split()))

    return data["tweet"]

In [7]:
preprocessing_txt=preprocessing(text)

In [8]:
preprocessing_txt

0    great love product
Name: tweet, dtype: object

In [9]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
tokens = vocab[0].tolist()

In [10]:
vocab

,0
0,test
1,android
2,app
3,beauti
4,cute
...,...
1140,rest
1141,disney
1142,develop
1143,e


In [11]:
def vectorized(ds,vocabulary):
    vactorized_lst=[]

    for senstence in ds:
        senstence_lst = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in senstence.split():
                senstence_lst[i]=1

        vactorized_lst.append(senstence_lst)

    vactorized_lst_new = np.asarray(vactorized_lst,dtype=np.float32)

    return vactorized_lst_new
                

In [12]:
vectorized_txt =vectorized(preprocessing_txt,tokens)

In [13]:
vectorized_txt

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 1145), dtype=float32)

In [14]:
with open('../static/model/model.pickle','rb') as f:
    model = pickle.load(f)

In [15]:
model.predict(vectorized_txt)

array([0])